# sorted-computational-graph — faded example 3: Use the perm set to visit each node exactly once

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`. Running the beacon reports progress on the `Backprop: Sorted computation graph` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In topological sort, the `perm` set (keyed by `id()` since tensors aren't hashable by value) tracks fully processed nodes. Before appending a node to the result, we add its id to `perm`. On any subsequent visit attempt, we check `if id(cur) in perm: return` to skip already-processed nodes. This is what prevents shared nodes from appearing twice in the result.

## Faded exercise 3

Complete the `visit` function inside `topological_sort` so it skips already-processed nodes.

1. At the start of `visit(cur)`, check if the node was already processed.
2. If yes, return immediately.
3. Otherwise, proceed with DFS.

The blank step is the early-return guard for already-processed nodes.

**Fill in:** Return immediately if id(cur) is already in the perm set, preventing duplicate visits to shared nodes.

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        for child in get_children(cur):
            visit(child)
        perm.add(id(cur))
        result.append(cur)
    visit(node)
    return result

def get_children(n):
    if n.recipe is None:
        return []
    return list(n.recipe.parents.values())

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [a, b])

result = topological_sort(c, get_children)[::-1]
names = [n.name for n in result]
print('Result:', names)
print('Length (should be 3):', len(result))
assert len(result) == 3


def _test():
    class FakeRecipe:
        def __init__(self, parents):
            self.parents = {i: p for i, p in enumerate(parents)}
    class FakeTensor:
        def __init__(self, name, parents=None):
            self.name = name
            self.recipe = FakeRecipe(parents) if parents else None

    def get_children(n):
        if n.recipe is None:
            return []
        return list(n.recipe.parents.values())

    a = FakeTensor('a')
    b = FakeTensor('b', [a])
    c = FakeTensor('c', [a, b])

    result = topological_sort(c, get_children)
    # Should be exactly 3 distinct nodes
    assert len(result) == 3, f'Expected 3 nodes, got {len(result)} — dedup failed'
    node_ids = [id(n) for n in result]
    assert len(set(node_ids)) == 3, 'All ids should be distinct — no duplicates'
    # Forward order: a before b before c
    idx = {id(n): i for i, n in enumerate(result)}
    assert idx[id(a)] < idx[id(c)], 'a must come before c in forward order'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        for child in get_children(cur):
            visit(child)
        perm.add(id(cur))
        result.append(cur)
    visit(node)
    return result

def get_children(n):
    if n.recipe is None:
        return []
    return list(n.recipe.parents.values())

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [a, b])

result = topological_sort(c, get_children)[::-1]
names = [n.name for n in result]
print('Result:', names)
print('Length (should be 3):', len(result))
assert len(result) == 3
```
</details>